<a href="https://colab.research.google.com/github/a-mhamdi/jlai/blob/main/Codes/Julia/Part-3/vae/vae.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VARIATIONAL AUTOENCODER (VAE)
---

**VAE** implemented in `Julia` using the `Flux.jl` library

In [1]:
versioninfo()  # 1.12.6

Julia Version 1.12.6
Commit 15346901f00 (2026-04-09 19:20 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: Linux (x86_64-linux-gnu)
  CPU: 2 × Intel(R) Xeon(R) CPU @ 2.00GHz
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, skylake-avx512)
  GC: Built with stock GC
Threads: 2 default, 1 interactive, 2 GC (on 2 virtual cores)
Environment:
  LD_LIBRARY_PATH = /usr/lib64-nvidia
  JULIA_NUM_THREADS = auto


Manage project dependencies.

In [2]:
pkgs = """[deps]
BSON = "fbb218c0-5317-5bc6-957e-2ee96dd4b1f0"
CUDA = "052768ef-5323-5732-b1bb-66c8b64840ba"
CairoMakie = "13f3f980-e62b-5c42-98c6-ff1f3baf88f0"
DrWatson = "634d3b9d-ee7a-5ddf-bec9-22491ea816e1"
Flux = "587475ba-b771-5e3f-ad9e-33799f191a9c"
ImageInTerminal = "d8c32880-2388-543b-8c61-d9f865259254"
ImageShow = "4e3cecfd-b093-5904-9786-8bbb286a6a31"
JLD2 = "033835bb-8acc-5ee8-8aae-3f567f8a3819"
MLDatasets = "eb30cadb-4394-5ae3-aed4-317e484a6458"
Markdown = "d6f4376e-aef5-505a-96c1-9c027394607a"
ProgressMeter = "92933f4c-e287-5a05-a399-4b506db050ca"
Statistics = "10745b16-79ce-11e8-11f9-7d13ad32a3b2"
cuDNN = "02a925ec-e4fe-4b08-9a7e-0d78e3d38ccd"
"""

open("Project.toml", "w") do file
    write(file, pkgs)
end

657

Activate the project environment and instantiate the listed packages.

In [3]:
_ = begin
  import Pkg;
  Pkg.activate(".");
  Pkg.instantiate();
end

  Activating project at `/content`
Precompiling packages...
              ✗ libwebp_jll
              ✗ WebP
  0 dependencies successfully precompiled in 9 seconds. 500 already precompiled.


Display the status of the packages in the active project environment.

In [4]:
Pkg.status()

Status `/content/Project.toml`
  [fbb218c0] BSON v0.3.9
  [052768ef] CUDA v6.5.1
  [13f3f980] CairoMakie v0.15.17
  [634d3b9d] DrWatson v2.19.1
  [587475ba] Flux v0.16.15
  [d8c32880] ImageInTerminal v0.5.4
  [4e3cecfd] ImageShow v0.3.8
  [033835bb] JLD2 v0.6.7
  [eb30cadb] MLDatasets v0.7.22
  [92933f4c] ProgressMeter v1.13.0
  [10745b16] Statistics v1.11.5
  [02a925ec] cuDNN v6.5.1
  [d6f4376e] Markdown v1.11.0


Import the Flux library and some of its submodules.

In [5]:
using Flux # v"0.16.5"
using Flux: DataLoader

Import the `CUDA` library for **GPU** acceleration.

In [6]:
using CUDA

In [7]:
CUDA.functional()

true

Import the `ProgressMeter` library for displaying progress during training.

In [8]:
using ProgressMeter: Progress, next!

In [9]:
import ProgressMeter
# ProgressMeter.ijulia_behavior(:append)

Load the **MNIST** dataset.

In [10]:
using MLDatasets
d = MNIST()

dataset MNIST:
  metadata  =>    Dict{String, Any} with 3 entries
  split     =>    :train
  features  =>    28×28×60000 Array{Float32, 3}
  targets   =>    60000-element Vector{Int64}

Let's define a mutable struct `HyperParams` to hold hyperparameters for the **VAE** model.

In [11]:
Base.@kwdef mutable struct HyperParams
    η = 3f-3                        # Learning rate
    λ = 1f-2                        # Regularization parameter
    batchsize = 256                 # Batch size
    epochs = 8                      # Number of epochs
    split = :train                  # Split data into `train` and `test`
    input_dim = 28*28               # Input dimension
    hidden_dim = 1024               # Hidden dimension
    latent_dim = 32                 # Latent dimension
    save_path = "Output"            # Results folder
end

HyperParams

Load and prepare the **MNIST** data for training and testing.

In [12]:
function get_data(; kws...)
    args = HyperParams(; kws...);
    # Split data
    data = MNIST(split=args.split);
    X = reshape(data.features, (args.input_dim, :));
    return DataLoader(X; batchsize=args.batchsize, shuffle=true)
end

get_data (generic function with 1 method)

Load the training and testing data using the `get_data` function.

In [13]:
train_loader = get_data();
test_loader = get_data(split=:test);

This struct defines the encoder network for the **VAE**.

In [14]:
struct Encoder
    hidden
    μ
    log_σ2
end

Make `Encoder` struct compatible with `Flux`'s automatic differentiation.

In [15]:
Flux.@layer Encoder

Construct an `Encoder` model with specified input, hidden, and latent dimensions.

In [16]:
Encoder(input_dim::Int, hidden_dim::Int, latent_dim::Int) = Encoder(
    Dense(input_dim, hidden_dim, tanh),   # hidden
    Dense(hidden_dim, latent_dim),        # μ
    Dense(hidden_dim, latent_dim),        # log_σ2
    )

Encoder

Define the forward pass for the encoder, returning the mean and log variance of the latent distribution.

In [17]:
function (encoder::Encoder)(x)
    h = encoder.hidden(x)
    encoder.μ(h), encoder.log_σ2(h)
end


Construct a decoder model with specified input, hidden, and output dimensions.

In [18]:
Decoder(input_dim::Int, hidden_dim::Int, latent_dim::Int) = Chain(
    Dense(latent_dim, hidden_dim, tanh),
    Dense(hidden_dim, input_dim)
    )

Decoder (generic function with 1 method)

Define the forward pass for the **VAE**, encoding the input, sampling from the latent space, and decoding to reconstruct the input.

In [19]:
using Random

In [20]:
function vae(x, enc, dec)
    # Encode `x` into the latent space
    μ, log_σ2 = enc(x)
    # `z` is a sample from the latent distribution
    ε = similar(μ)
    randn!(ε)
    z = @. μ + ε * exp(.5f0 * log_σ2)
    # Decode the latent representation into a reconstruction of `x`
    x̂ = dec(z)
    # Return μ, log_σ2 and x̂
    μ, log_σ2, x̂
end

vae (generic function with 1 method)

Calculate the loss for the **VAE**, including the reconstruction loss, **KL** divergence loss, and **L2** regularization.

In [21]:
function l(x, enc, dec, λ)
    μ, log_σ2, x̂ = vae(x, enc, dec)
    n = size(x)[end]

    # Reconstruction loss: how well the decoder reproduces the input
    recon_loss = Flux.Losses.logitbinarycrossentropy(x̂, x, agg=sum) / n

    # KL divergence: how close the latent distribution is to the standard normal
    kl_loss = .5f0 * sum(@. exp(log_σ2) + μ^2 - 1f0 - log_σ2) / n

    # Correct L2 regularization: sum over individual numeric parameter arrays
    reg_loss = λ * sum(sum(abs2, p) for p in Flux.params(dec))

    # Total loss
    return recon_loss + kl_loss + reg_loss
end

l (generic function with 1 method)

Train the **VAE** model for a specified number of epochs using the `Adam` optimizer.

In [22]:
function train(; kws...)
    args = HyperParams(; kws...)

    # Guarantee GPU execution by converting the networks to GPU types
    enc_mdl = Encoder(args.input_dim, args.hidden_dim, args.latent_dim) |> CUDA.cu
    dec_mdl = Decoder(args.input_dim, args.hidden_dim, args.latent_dim) |> CUDA.cu

    # Display network types to confirm they are indeed CuArrays (GPU)
    println("Encoder type: ", typeof(enc_mdl.hidden.weight))
    println("Decoder type: ", typeof(dec_mdl[1].weight))

    # Check GPU's memory status
    println(CUDA.memory_info())

    # ADAM optimizers
    opt_enc = Flux.setup(Adam(args.η), enc_mdl)
    opt_dec = Flux.setup(Adam(args.η), dec_mdl)

    first_batch = true
    for epoch in 1:args.epochs
        printstyled("\t***\t === EPOCH $(epoch) === \t*** \n", color=:magenta, bold=true)
        progress = Progress(length(train_loader))
        for X in train_loader
            # Explicitly move the batch to the GPU
            X_gpu = CUDA.cu(X)
            if first_batch
                println("Batch array type on GPU: ", typeof(X_gpu))
                first_batch = false
            end
            loss, grads = Flux.withgradient(enc_mdl, dec_mdl) do enc, dec
                l(X_gpu, enc, dec, args.λ)
            end
            Flux.update!(opt_enc, enc_mdl, grads[1])  # Upd `encoder` params
            Flux.update!(opt_dec, dec_mdl, grads[2])  # Upd `decoder` params
            next!(progress; showvalues=[(:loss, loss)])
        end
    end

    enc_mdl, dec_mdl
end

train (generic function with 1 method)

Initiate the training process for the **VAE** model.

In [ ]:
enc_model, dec_model = train()

┌ Warning: ProgressMeter by default refresh meters with additional information in IJulia via `IJulia.clear_output`, which clears all outputs in the cell. 
│  - To prevent this behaviour, do `ProgressMeter.ijulia_behavior(:append)`. 
│  - To disable this warning message, do `ProgressMeter.ijulia_behavior(:clear)`.
└ @ ProgressMeter ~/.julia/packages/ProgressMeter/OPrdc/src/ProgressMeter.jl:651
Progress:  62%|█████████████████████████▎               |  ETA: 0:04:22
   loss: 135.23404

In [ ]:
#= Save the model

using DrWatson: struct2dict
using BSON

args = HyperParams()
mdl_path = joinpath(args.save_path, "vae.bson")
let args=struct2dict(args)
    BSON.@save mdl_path enc_model dec_model args
    @info "Model saved to $(mdl_path)"
end
=#